<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Key signal distributions

I inspected the distributions of the two signals used in my baseline idea: content staleness and search visibility.

Staleness is concentrated in the 0–30 day bucket, but there is also a substantial group of pages in the 91–180 day bucket. Search visibility is spread across all five impression buckets, with the largest groups in the 101–1K and 1K–10K ranges.

The distributions are therefore not uniform and contain some heavy-tail behavior, especially for search impressions.

In [17]:
import pandas as pd

DATA_PATH = "/content/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [18]:
# Section 1 — Distributions

import numpy as np

# Create the two derived buckets used for signal auditing

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 60, 90, 180, 365, np.inf],
    labels=[
        "0-30 days",
        "31-60 days",
        "61-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 10, 100, 1000, 10000, np.inf],
    labels=[
        "0-10 impressions",
        "11-100 impressions",
        "101-1K impressions",
        "1K-10K impressions",
        "10K+ impressions"
    ]
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nStaleness distribution:")
staleness_dist = (
    df["staleness_bucket"]
    .value_counts(sort=False)
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

print(staleness_dist)

print("\nSearch visibility distribution:")
visibility_dist = (
    df["visibility_bucket"]
    .value_counts(sort=False)
    .rename_axis("visibility_bucket")
    .reset_index(name="n")
)

print(visibility_dist)

Rows: 30000
Columns: 46

Staleness distribution:
  staleness_bucket      n
0        0-30 days  20480
1       31-60 days    128
2       61-90 days     47
3      91-180 days   9171
4     181-365 days    169
5        365+ days      5

Search visibility distribution:
    visibility_bucket     n
0    0-10 impressions  3879
1  11-100 impressions  4127
2  101-1K impressions  8485
3  1K-10K impressions  9907
4    10K+ impressions  3602


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal 1 — Staleness

In [19]:
# Signal Test #1 — Staleness vs declining proxy

staleness_signal_test = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

staleness_signal_test

,staleness_bucket,n,median_impressions,median_ctr
0,0-30 days,20480,470.0,0.040
1,31-60 days,128,699.5,0.015
2,61-90 days,47,187.0,0.000
3,91-180 days,9171,1692.0,0.100
4,181-365 days,169,16.0,0.000
5,365+ days,5,2.0,0.000


**Verdict: MIXED**

Staleness varies substantially across the content set, but the relationship with performance is not consistently negative. In particular, pages that are 91–180 days old have higher median impressions and CTR than the 0–30 day group, while very old pages have much lower values. Therefore, staleness is useful for prioritization, but it does not by itself prove that a page is underperforming or needs a refresh.

### Signal 2 — Search visibility

In [20]:
# Signal Test #2 — Search visibility vs engagement

visibility_signal_test = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          median_avg_position=("avg_position", "median")
      )
      .reset_index()
)

visibility_signal_test

,visibility_bucket,n,median_ctr,median_avg_position
0,0-10 impressions,3879,0.00,4.3
1,11-100 impressions,4127,0.00,10.3
2,101-1K impressions,8485,0.00,15.8
3,1K-10K impressions,9907,0.17,11.3
4,10K+ impressions,3602,0.23,7.5


**Verdict: CONFIRMED**

Search visibility varies substantially across the dataset. Higher-impression content represents a larger potential audience, so impressions are useful for prioritizing pages where a successful refresh could have greater impact.

### Signal 3 — Click-through rate (CTR)

In [21]:
# Signal Test #3 — CTR distribution

df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-np.inf, 0, 0.05, 0.2, 1, np.inf],
    labels=[
        "0%",
        "0-0.05%",
        "0.05-0.2%",
        "0.2-1%",
        "1%+"
    ]
)

ctr_signal_test = (
    df.groupby("ctr_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median")
      )
      .reset_index()
)

ctr_signal_test

,ctr_bucket,n,median_impressions
0,0%,13212,74.0
1,0-0.05%,1207,5382.0
2,0.05-0.2%,5886,3086.5
3,0.2-1%,8006,2855.5
4,1%+,1689,98.0


**Verdict: MIXED**

CTR varies substantially across the content set, but its relationship with search visibility is not monotonic. In particular, the 1%+ CTR bucket has a much lower median impression count than the 0.2–1% bucket. CTR is therefore a useful descriptive signal, but this test does not provide strong evidence that higher CTR should increase baseline priority.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Staleness

**FlyRank flag:** Refresh / staleness flag

**Signal:** `days_since_last_update`

**Question:** Does the data support the assumption that stale content is a meaningful signal for a refresh action?

In [22]:
# Flag-linked test — staleness

flag_linked_staleness = (
    df.assign(
        stale_90_plus=df["days_since_last_update"] >= 90
    )
    .groupby("stale_90_plus")
    .agg(
        n=("content_id", "size"),
        median_impressions=("impressions_90d", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

flag_linked_staleness

,stale_90_plus,n,median_impressions,median_ctr
0,False,20655,472.0,0.04
1,True,9345,1621.0,0.10


**Verdict: MIXED**

The data supports staleness as a useful prioritization signal because 9,345 pages are at least 90 days since their last update and this group has substantial search visibility. However, stale pages have higher median impressions and CTR than non-stale pages, so staleness alone does not indicate poor current performance. The refresh assumption is therefore only partially supported.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

Content teams should prioritize **stale pages with high search visibility**, since these pages have both a strong refresh opportunity and potentially greater impact. The rule is a **prioritization aid, not proof that a refresh will improve performance**, so high-scoring pages should still be reviewed before action.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.